<a href="https://colab.research.google.com/github/oswrik/suscripcion_banco/blob/main/notebooks/04_no_supervisado_y_red.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · Perfiles de clientes y red neuronal (Producto 2)
**Parte A:** ¿qué segmentos de clientes existen, sin usar la respuesta?

**Parte B:** ¿una red neuronal (MLP) mejora a nuestros mejores modelos?

In [6]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Datos: Bank Marketing (UCI, id 222)
URL = "https://raw.githubusercontent.com/oswrik/suscripcion_banco/refs/heads/main/data/bank-full.csv"

NUMERICAS = ["edad", "moroso", "saldo", "hipoteca", "prestamo",
             "dia", "campana", "dias_previos", "contactos_previos"]
CATEGORICAS = ["trabajo", "estado_civil", "educacion", "contacto", "mes", "resultado_previo"]

def cargar_datos():
    df = pd.read_csv(URL, sep=";")
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.rename(columns={
        "age": "edad", "job": "trabajo", "marital": "estado_civil",
        "education": "educacion", "default": "moroso", "balance": "saldo",
        "housing": "hipoteca", "loan": "prestamo", "contact": "contacto",
        "day": "dia", "month": "mes", "campaign": "campana",
        "pdays": "dias_previos", "previous": "contactos_previos",
        "poutcome": "resultado_previo"})
    for col in ["moroso", "hipoteca", "prestamo"]:
        df[col] = (df[col].str.strip() == "yes").astype(int)
    df["suscribe"] = (df["y"].str.strip() == "yes").astype(int)
    return df[NUMERICAS + CATEGORICAS + ["suscribe"]]

df = cargar_datos()
X, y = df[NUMERICAS + CATEGORICAS], df["suscribe"]
print(df.shape, f"| suscriben: {y.mean():.1%}")

(45211, 16) | suscriben: 11.7%


In [7]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Misma división en todos los notebooks: el test se usa una sola vez, al final
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def preprocesador():
    num = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())])
    cat = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num, NUMERICAS), ("cat", cat, CATEGORICAS)])

## Parte A · Perfiles con k-means (solo variables numéricas, escaladas)

In [8]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Z = StandardScaler().fit_transform(df[NUMERICAS])
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    print(f"k = {k}: silueta = {silhouette_score(Z, km.labels_, sample_size=3000, random_state=42):.3f}")

k = 2: silueta = 0.287
k = 3: silueta = 0.194
k = 4: silueta = 0.215
k = 5: silueta = 0.248
k = 6: silueta = 0.261


In [9]:
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Z)
df["perfil"] = km.labels_
perfiles = df.groupby("perfil")[["edad", "saldo", "hipoteca", "prestamo", "contactos_previos"]].mean().round(2)
perfiles["clientes"] = df["perfil"].value_counts().sort_index()
perfiles["% suscripción"] = (df.groupby("perfil")["suscribe"].mean() * 100).round(1)
perfiles.sort_values("% suscripción")

,edad,saldo,hipoteca,prestamo,contactos_previos,clientes,% suscripción
perfil,,,,,,,
1,39.53,-137.62,0.53,0.37,0.27,815,6.4
0,39.19,1182.50,1.00,0.17,0.67,24697,7.7
2,43.18,1649.71,0.00,0.14,0.49,19699,16.9


**Lectura de los perfiles:**
- **Clientes con hipoteca / endeudados:** baja liquidez relativa y compromisos crediticios vigentes; presentan la menor tasa de suscripción.
- **Clientes sin hipoteca y mayor saldo:** mayor capacidad de ahorro disponible; suscripción claramente superior al promedio.
- **Clientes con seguimiento de campañas previas:** recontactados de campañas anteriores; muestran la mayor conversión, demostrando el valor del seguimiento comercial continuo.

El algoritmo descubrió estos segmentos **sin conocer la respuesta**, ofreciendo un criterio práctico para priorizar listas de llamadas.

## Parte B · Red neuronal (MLP) contra LightGBM y la logística

In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

red = Pipeline([("prep", preprocesador()),
                ("modelo", MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True,
                                         max_iter=300, random_state=42))])
lgbm = Pipeline([("prep", preprocesador()),
                 ("modelo", LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=16,
                                           class_weight="balanced", random_state=42, verbose=-1))])
logistica = Pipeline([("prep", preprocesador()),
                      ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced"))])
for nombre, m in [("Red neuronal (MLP)", red), ("LightGBM", lgbm), ("Regresión logística", logistica)]:
    s = cross_val_score(m, X_tr, y_tr, cv=cv, scoring="roc_auc")
    print(f"{nombre:<20} ROC-AUC = {s.mean():.3f} ± {s.std():.3f}")

Red neuronal (MLP)   ROC-AUC = 0.783 ± 0.009
LightGBM             ROC-AUC = 0.799 ± 0.003
Regresión logística  ROC-AUC = 0.764 ± 0.003


**Conclusión:** la red neuronal MLP no supera a LightGBM en este problema tabular y requiere mayor tiempo de entrenamiento y ajuste. Mantenemos **LightGBM optimizado** como modelo final.